In [ ]:
# Example 1: top-level package metadata + public surface inspection.
import jupyter_server_documents

version = getattr(jupyter_server_documents, '__version__', None)
print(f"version: {version}")

public_names = [n for n in dir(jupyter_server_documents) if not n.startswith('_')]
print(f"public top-level names ({len(public_names)}):")
for n in public_names[:25]:
    print(f"  - {n}")

assert hasattr(jupyter_server_documents, '__path__'), 'package should expose __path__'


In [ ]:
# Example 2: jupyter-server extension entrypoint discovery.
# jupyter_server_documents is a jupyter_server extension; it must expose _jupyter_server_extension_points.
import jupyter_server_documents

ext_points_fn = getattr(jupyter_server_documents, '_jupyter_server_extension_points', None)
print(f"_jupyter_server_extension_points present: {ext_points_fn is not None}")

if ext_points_fn is not None:
    points = ext_points_fn()
    print(f"extension points returned: {points}")
    assert isinstance(points, list), 'extension points should be a list'
    for p in points:
        assert isinstance(p, dict), 'each entry should be a dict'
        assert 'module' in p, 'each entry should declare a module'

print('extension entrypoint contract OK')


In [ ]:
# Example 3: ExtensionApp class discoverable + instantiable.
import jupyter_server_documents
from jupyter_server.extension.application import ExtensionApp

# Find any ExtensionApp subclass declared by the package.
candidates = []
for name in dir(jupyter_server_documents):
    obj = getattr(jupyter_server_documents, name)
    if isinstance(obj, type) and issubclass(obj, ExtensionApp) and obj is not ExtensionApp:
        candidates.append((name, obj))

print(f"ExtensionApp subclasses found: {[n for n, _ in candidates]}")

if candidates:
    name, cls = candidates[0]
    inst = cls()
    print(f"instantiated {name}: {inst!r}")
    assert isinstance(inst, ExtensionApp)
else:
    print('no direct ExtensionApp subclass at top level (may live in submodules)')


In [ ]:
# Example 4: pycrdt + jupyter_ydoc integration is reachable.
# jupyter_server_documents depends on these for CRDT-backed collaborative documents.
import jupyter_ydoc
import pycrdt

# Construct a YDoc through pycrdt and exercise a basic op.
doc = pycrdt.Doc()
text = pycrdt.Text()
doc['content'] = text
text += 'hello, collab world'

print(f"pycrdt.Doc: {doc!r}")
print(f"text content: {str(text)!r}")

assert str(text) == 'hello, collab world'
print(f"jupyter_ydoc version: {getattr(jupyter_ydoc, '__version__', 'unknown')}")


In [ ]:
# Example 5: extension is enabled via jupyter_server's extension manager.
from jupyter_server.extension.manager import ExtensionPackage

pkg = ExtensionPackage(name='jupyter_server_documents', enabled=True)
print(f"ExtensionPackage: {pkg!r}")
print(f"  name: {pkg.name}")
print(f"  enabled: {pkg.enabled}")
print(f"  metadata: {pkg.metadata}")

# Each registered point must declare a module name.
for point in pkg.extension_points.values():
    print(f"  point module: {point.module_name}")
    assert point.module_name.startswith('jupyter_server_documents')

assert pkg.name == 'jupyter_server_documents'
assert pkg.enabled is True
print('extension successfully registered with jupyter_server')
